# Imports

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score, accuracy_score

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb

In [ ]:
df = pd.read_csv('../data/df_stats.csv', sep=';', dtype={'Code_INSEE': str})
df_train = df[df['Année'] == 2022].copy()
display(df_train.head(5))
print(f"Dataset chargé : {df_train.shape[0]} lignes, {df_train.shape[1]} colonnes")

print(f"\nDistribution de la cible (vote politique) :")
print(df_train['Résultat'].value_counts())
print(f"\nProportions :")
print(df_train['Résultat'].value_counts(normalize=True).round(4) * 100)

print('\nColonnes :')
print(df_train.columns)

# df_null = df[df.isna().any(axis=1)]
# df = df.dropna()
# print(f"Valeurs manquantes supprimées, df final : {df.shape[0]} lignes, {df.shape[1]} colonnes")

# Feature engineering

In [ ]:
# Ordinal encoding manuel de la variable cible
groups_dict = {'gauche': 0, 'centre': 1, 'droite': 2}
df_train['Résultat'] = df_train['Résultat'].replace(groups_dict).astype(int)


df_train = df_train.drop(columns=['Année','Libellé de la commune', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', '% gauche/Exp', '% centre/Exp', '% droite/Exp'])


def convert_columns_to_percentages(df, list_columns, divider_column):
    for col in list_columns:
        df[col] = df[col] / df[divider_column] * 100
    return df


# Transformation des genres, catégories socio-professionnelles, tranches d'âge et statuts maritaux en pourcentages de la population active
columns_to_convert = [
    'Hommes', 
    'Femmes', 
    'Agriculteurs', 
    'Artisans', 
    'Cadres', 
    'Intermédiaires', 
    'Employés', 
    'Ouvriers',
    'Retraités',
    'Etudiants',
    'Inactifs',
    '15-24 ans',
    '25-39 ans',
    '40-54 ans',
    '55-64 ans',
    '65-79 ans',
    '80 ans et +',
    'Mariés',
    'Pacsés',
    'Concubinage',
    'Veufs',
    'Divorcés',
    'Célibataires',
]

df_train = convert_columns_to_percentages(df_train, columns_to_convert, 'Population_active')



# Transformation des compositions de ménages en pourcentage de la population avec enfants

columns_to_convert_household = [
    'Personne seule',
    'Homme seul',
    'Femme seule',
    'Colocation',
    'Famille',
    'Famille monoparentale',
    'Couple sans enfant',
    'Couple avec enfants',
]
df_train = convert_columns_to_percentages(df_train, columns_to_convert_household, 'Population avec enfants')

display(df_train)



# Modélisation

## Split test-train

In [ ]:
X = df_train.drop(columns=['Code_INSEE', 'Résultat'])
y = df_train['Résultat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("✓ Données préparées")
print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")

## XGBoost + SMOTE

In [ ]:
xgb_smote = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        objective='multi:softmax',
        eval_metric='mlogloss'
    ))
])


xgb_smote.fit(X_train, y_train)


y_pred_test = xgb_smote.predict(X_test)
y_pred_train = xgb_smote.predict(X_train)


train_score = xgb_smote.score(X_train, y_train)
test_score = xgb_smote.score(X_test, y_test)
gap = abs(train_score - test_score)

print(f"  Gap   : {gap:.2%}\n")


if gap > 0.10:
    print("⚠️ OVERFITTING détecté !")
elif train_score < 0.75:
    print("⚠️ UNDERFITTING détecté !")
else:
    print("✅ Bon équilibre\n")


groups = ['gauche', 'centre', 'droite']


# Métriques - train
print('\n---RÉSULTATS DE XGBOOST AVEC SMOTE SUR LE TRAIN\n')
print(classification_report(y_train, y_pred_train, target_names=groups))

# Cross validation - Random Search

In [ ]:
param_distributions = {
    'classifier__learning_rate': [0.1, 0.2],
    'classifier__n_estimators': [200, 250, 300],
    'classifier__subsample': [0.8, 0.5],
    'classifier__colsample_bytree': [0.8, 0.5]
}
# max_depth reste à 5 sinon le modèle est en overfitting


print("Lancement de Random Search...")
random_search = RandomizedSearchCV(
    xgb_smote,
    param_distributions,
    n_iter=50,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)


# Décommenter la ligne pour lancer le Random Search
# random_search.fit(X_train, y_train)


print("\n✓ Random Search terminé !")
print(f"\nMeilleurs hyperparamètres :")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nMeilleur score (CV) : {random_search.best_score_:.2%}")


# Évaluation sur les données de test

In [ ]:
print('\n---RÉSULTATS DE XGBOOST AVEC SMOTE SUR LE TEST\n')
print(classification_report(y_test, y_pred_test, target_names=groups))

| Métrique | Train | Test |
| :------- | :---- | :--- |
| Accuracy | 75 % | 67 % |
| Precision avg | 80 % | 72 % |
| Recall | 75 % | 67 % |
| F1-score | 77 % | 69 % |


- accuracy : 75 % des prédictions sont correctes
- precision : parmi les prédictions 'gauche', 49 % sont correctes (idem pour chaque groupe)
- recall : parmi toutes les communes à gauche, 73 % sont prédites correctement (idem pour chaque groupe)
- f1-score : moyenne entre precision et recall


## Tests automatisés

In [ ]:
def test_model_performance(trained_model, test_data_X, test_data_y):
    predictions = trained_model.predict(test_data_X)

    current_accuracy = accuracy_score(test_data_y, predictions)
    current_f1 = f1_score(test_data_y, predictions, average='weighted')

    MIN_EXPECTED_ACCURACY = 0.65
    MIN_EXPECTED_F1 = 0.65

    assert current_accuracy >= MIN_EXPECTED_ACCURACY, f"Accuracy {current_accuracy} < {MIN_EXPECTED_ACCURACY}"
    assert current_f1 >= MIN_EXPECTED_F1, f"F1-score {current_f1} < {MIN_EXPECTED_F1}"



def test_model_cv_performance_and_variance(model, training_data_X, training_data_y):
    cv_scores = cross_val_score(model, training_data_X, training_data_y, cv=5, scoring='f1_score')

    mean_cv_f1 = np.mean(cv_scores)
    std_cv_f1 = np.std(cv_scores)

    MIN_EXPECTED_CV_F1 = 0.65
    MAX_EXPECTED_CV_STD = 0.05

    print(f"\\nValidation croisée - Accuracy moyenne: {mean_cv_f1:.3f}")
    print(f"Validation croisée - Écart-type des accuracies: {std_cv_f1:.3f}")

    assert mean_cv_f1 >= MIN_EXPECTED_CV_F1, f"Accuracy moyenne en CV {mean_cv_f1} < {MIN_EXPECTED_CV_F1}"
    assert std_cv_f1 <= MAX_EXPECTED_CV_STD, f"Écart-type en CV {std_cv_f1} > {MAX_EXPECTED_CV_STD}, variance trop élevée."
